# Environment Setup & Verification
**Project:** Nemotron Tabular Classification  
Run this notebook after following `SETUP.md` to verify everything is installed correctly.

Every cell should run without errors before you start the weekly notebooks.

## 1. Verify Python Version

In [ ]:
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 11), "Python 3.11+ required"
print("✓ Python version OK")

## 2. Verify All Required Packages

In [ ]:
import importlib

required = [
    "pandas", "numpy", "sklearn", "xgboost",
    "datasets", "huggingface_hub",
    "matplotlib", "seaborn",
    "requests", "ipykernel"
]

all_ok = True
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  ✗ {pkg} — NOT INSTALLED")
        all_ok = False

print()
if all_ok:
    print("✓ All packages installed correctly")
else:
    print("✗ Some packages missing — run: pip install -r requirements.txt")

## 3. Verify Package Versions

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import xgboost as xgb
import matplotlib
import seaborn as sns

print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"sklearn:    {sklearn.__version__}")
print(f"xgboost:    {xgb.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn:    {sns.__version__}")
print()
print("✓ Package versions OK")

## 4. Verify GPU (CUDA)

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        gpu_info = result.stdout.strip()
        print(f"GPU detected: {gpu_info}")
        print("✓ GPU available")
    else:
        print("✗ nvidia-smi failed — GPU may not be available")
except Exception as e:
    print(f"✗ Could not detect GPU: {e}")

## 5. Verify Ollama is Running

In [ ]:
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✓ Ollama is running")
    print(f"  Available models: {models}")

    # Check for Nemotron models
    nemotron_models = [m for m in models if "nemotron" in m.lower()]
    if nemotron_models:
        print(f"  ✓ Nemotron models found: {nemotron_models}")
    else:
        print("  ✗ No Nemotron models found")
        print("    Run: ollama pull nemotron-3-nano:4b")
except Exception as e:
    print(f"✗ Ollama not reachable: {e}")
    print("  Make sure Ollama is running: ollama serve")

## 6. Verify Dataset Access

In [ ]:
import os
from datasets import load_dataset

# Check if sample already exists
sample_path = "../data/personas_sample_5000.csv"
if os.path.exists(sample_path):
    import pandas as pd
    sample = pd.read_csv(sample_path)
    print(f"✓ Sample file found: {sample_path}")
    print(f"  Rows: {len(sample)}")
    print(f"  Columns: {list(sample.columns)}")
else:
    print("Sample file not found — checking HuggingFace access...")
    try:
        # Just load metadata without downloading full dataset
        ds = load_dataset(
            "nvidia/Nemotron-Personas-USA",
            split="train",
            streaming=True
        )
        # Peek at first row
        first = next(iter(ds))
        print(f"✓ HuggingFace dataset accessible")
        print(f"  Columns: {list(first.keys())}")
        print("  Run week1_baselines.ipynb to download and generate the sample")
    except Exception as e:
        print(f"✗ Cannot access dataset: {e}")
        print("  Check internet connection")

## 7. Verify Folder Structure

In [ ]:
import os

required_folders = ["../data", "../models", "../notebooks", "../src", "../results"]
required_src     = ["serialize.py", "prompts.py", "parse_response.py",
                    "infer_local.py", "infer_fds.py", "evaluate.py"]

print("Folder structure:")
for folder in required_folders:
    exists = os.path.exists(folder)
    print(f"  {'✓' if exists else '✗'} {folder}")

print("\nsrc/ files:")
for fname in required_src:
    path = f"../src/{fname}"
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {fname}")

## 8. Quick Inference Test

In [ ]:
import requests
import time

OLLAMA_URL = "http://localhost:11434/v1/chat/completions"

# Try to detect which Nemotron model is available
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    nemotron = next((m for m in models if "nemotron" in m.lower()), None)

    if not nemotron:
        print("✗ No Nemotron model found — skipping inference test")
        print("  Run: ollama pull nemotron-3-nano:4b")
    else:
        print(f"Testing inference with {nemotron}...")
        payload = {
            "model": nemotron,
            "messages": [
                {"role": "system", "content": "You are an education classifier. Reply with college or not_college only."},
                {"role": "user",   "content": "A 35-year-old female working as a software engineer. college or not_college?"}
            ],
            "max_tokens": 256,
            "temperature": 0.1,
            "stream": False,
        }

        start = time.time()
        response = requests.post(OLLAMA_URL, json=payload, timeout=60).json()
        elapsed = time.time() - start

        content = response["choices"][0]["message"]["content"]
        print(f"✓ Inference successful")
        print(f"  Model:    {nemotron}")
        print(f"  Response: {content[:100]}")
        print(f"  Time:     {elapsed*1000:.0f}ms")

except Exception as e:
    print(f"✗ Inference test failed: {e}")
    print("  Make sure Ollama is running: ollama serve")

## 9. Setup Summary

In [ ]:
print("=" * 45)
print("  SETUP VERIFICATION COMPLETE")
print("=" * 45)
print()
print("If all cells above show ✓ you are ready to run:")
print()
print("  1. notebooks/week1_baselines.ipynb")
print("  2. notebooks/week2_prompts.ipynb")
print("  3. notebooks/week3_nano4b_results.ipynb")
print("  4. notebooks/week4_nano30b_results.ipynb")
print("  5. notebooks/week5_analysis.ipynb")
print()
print("See SETUP.md for full instructions and troubleshooting.")